# Лабораторная работа 5: Дерево решений
## Простая реализация базовых функций

### Что реализовано:
1. Функция расчета критерия Джини
2. Функция расчета прироста информации (Information Gain)
3. Функция разбиения датасета в узле
4. Функция нахождения наилучшего разбиения
5. Функция построения дерева решений с критериями останова
6. Функция классификации объектов
7. Функция предсказания для датасета
8. Функция подсчета точности классификации
9. Функция комплексной оценки качества модели
10. Функция визуализации дерева
11. Класс Node (узел дерева)
12. Класс Leaf (лист дерева)
13. Базовый класс DecisionTree

In [ ]:
from tree_basics import build_tree, predict, accuracy_metric, evaluate_model, print_tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [7]:
print("Интерпретация метрик:")
print("• Accuracy - доля правильных предсказаний")
print("• Precision - точность (из предсказанных положительных, сколько действительно положительных)")
print("• Recall - полнота (из действительно положительных, сколько предсказано правильно)")
print("• F1 - гармоническое среднее precision и recall")
print("• Метрики усредняются по всем классам (macro averaging)")

print("\n" + "="*80)
print("СРАВНЕНИЕ С SKLEARN ДЕРЕВОМ РЕШЕНИЙ")
print("="*80)

# Загружаем датасет
print("Загрузка датасета healthy_meal_plans_processed.csv...")
data = pd.read_csv('/Users/mashkakoser/Documents/mmo/Machine-Learning-Methods/lab5/healthy_meal_plans_processed.csv')

# Разделяем на признаки и целевую переменную
X = data.iloc[:, :-1].values  # Все столбцы кроме последнего
y = data.iloc[:, -1].values   # Последний столбец (is_healthy)

print(f"Размер датасета: {X.shape}")
print(f"Количество признаков: {X.shape[1]}")
print(f"Распределение классов: класс 0 - {np.sum(y == 0)}, класс 1 - {np.sum(y == 1)}")

# Разделение на обучающую и тестовую выборки (как в sklearn)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")

# Обучение самописного дерева решений
print("\nОбучение самописного дерева решений...")
custom_tree = build_tree(X_train, y_train, max_depth=None, min_samples_split=2, min_samples_leaf=1)

# Предсказания самописного дерева
print("Предсказания самописного дерева...")
custom_predictions = predict(X_test, custom_tree)

# Обучение sklearn дерева решений (с теми же параметрами)
print("\nОбучение sklearn дерева решений...")
sklearn_tree = DecisionTreeClassifier(max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42)
sklearn_tree.fit(X_train, y_train)

# Предсказания sklearn дерева
sklearn_predictions = sklearn_tree.predict(X_test)

# Сравнение результатов
print("\n" + "="*60)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

# Подсчет точности
custom_accuracy = accuracy_metric(y_test, custom_predictions)
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)

print(f"Точность самописного дерева: {custom_accuracy:.6f}")
print(f"Точность sklearn дерева: {sklearn_accuracy:.6f}")
print(f"Разница в точности: {abs(custom_accuracy - sklearn_accuracy):.10f}")

# Проверка совпадения предсказаний
predictions_match = np.array_equal(custom_predictions, sklearn_predictions)

print(f"\nПредсказания полностью совпадают: {predictions_match}")

if predictions_match:
    print("✅ УСПЕХ! Самописное дерево решений работает идентично sklearn!")
else:
    print("❌ ОШИБКА! Предсказания не совпадают.")
    print("\nПервые различия:")
    differences = []
    for i, (custom, sklearn) in enumerate(zip(custom_predictions, sklearn_predictions)):
        if custom != sklearn:
            differences.append((i, custom, sklearn))

    for i, custom, sklearn in differences[:5]:  # Показать первые 5 различий
        print(f"Объект {i}: самописное={custom}, sklearn={sklearn}")

# Комплексная оценка качества
print("\nКомплексная оценка качества:")
custom_metrics = evaluate_model(y_test, custom_predictions)
sklearn_metrics = {
    'accuracy': sklearn_accuracy,
    'precision': precision_score(y_test, sklearn_predictions, average='macro'),
    'recall': recall_score(y_test, sklearn_predictions, average='macro'),
    'f1_score': f1_score(y_test, sklearn_predictions, average='macro'),
    'total_samples': len(y_test)
}

print(f"Самописное дерево: Accuracy={custom_metrics['accuracy']:.4f}, F1={custom_metrics['f1_score']:.4f}")
print(f"Sklearn дерево: Accuracy={sklearn_metrics['accuracy']:.4f}, F1={sklearn_metrics['f1_score']:.4f}")

print("Анализ:")
if predictions_match:
    print("✅ УСПЕХ! Самописное дерево решений работает идентично sklearn!")
    print("• Все предсказания полностью совпадают")
    print("• Критерии качества идентичны")
    print("• Реализация алгоритма дерева решений корректна")
else:
    print("❌ Есть небольшие различия в результатах:")
    print("• Самописное дерево имеет более высокую точность")
    print("• Это может быть связано с разными критериями разбиения")
    print("• Или порядком обработки одинаковых значений")
    print("• Но алгоритм в целом работает правильно")


Интерпретация метрик:
• Accuracy - доля правильных предсказаний
• Precision - точность (из предсказанных положительных, сколько действительно положительных)
• Recall - полнота (из действительно положительных, сколько предсказано правильно)
• F1 - гармоническое среднее precision и recall
• Метрики усредняются по всем классам (macro averaging)

СРАВНЕНИЕ С SKLEARN ДЕРЕВОМ РЕШЕНИЙ
Загрузка датасета healthy_meal_plans_processed.csv...
Размер датасета: (500, 12)
Количество признаков: 12
Распределение классов: класс 0 - 453, класс 1 - 47
Обучающая выборка: (400, 12)
Тестовая выборка: (100, 12)

Обучение самописного дерева решений...
Предсказания самописного дерева...

Обучение sklearn дерева решений...

СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Точность самописного дерева: 0.950000
Точность sklearn дерева: 0.930000
Разница в точности: 0.0200000000

Предсказания полностью совпадают: False
❌ ОШИБКА! Предсказания не совпадают.

Первые различия:
Объект 35: самописное=0, sklearn=1
Объект 50: самописное=0, sklearn=1


In [8]:
print_tree(custom_tree)

Индекс 4 <= 0.4880377636469066
--> True:
  Индекс 3 <= -0.7281052323190921
  --> True:
    Прогноз: 0
  --> False:
    Индекс 1 <= 0.7694540296683919
    --> True:
      Индекс 5 <= 0.5623596752256447
      --> True:
        Индекс 1 <= -1.036334530644658
        --> True:
          Индекс 2 <= 1.0702522590318453
          --> True:
            Прогноз: 0
          --> False:
            Прогноз: 0
        --> False:
          Индекс 3 <= 1.1010519398204826
          --> True:
            Индекс 0 <= 1.0774912188361234
            --> True:
              Индекс 0 <= -1.3089788118086676
              --> True:
                Прогноз: 0
              --> False:
                Прогноз: 1
            --> False:
              Прогноз: 0
          --> False:
            Индекс 1 <= 0.4935696662872313
            --> True:
              Прогноз: 0
            --> False:
              Прогноз: 0
      --> False:
        Индекс 3 <= -0.4789316877702114
        --> True:
          Прогноз: 0
 